# Physics-Informed Neural Network for Diffusion-Reaction Problems with Dead-Core Formation in Catalyst Slabs

**Paper:** Skrzypacz, P., Tangirbergen, K., Valdman, J. (2026). *Physics-Informed Neural Network for Diffusion-Reaction Problems with Dead-Core Formation in Catalyst Slabs.* arXiv:2606.02599 [math.NA].

**Carpeta origen:** `PINNs/1. mecanica de fluidos/Physics-Informed Neural Network for.pdf`

## Como se usan las PINNs en este paper

El paper resuelve un problema de frontera libre: la ecuacion de difusion-reaccion adimensional en una lamina catalitica (Eq. 7-9)

$$u''=\phi^2 u^n \quad \text{en } (0,1), \qquad u'(0)=0,\quad u(1)=1$$

con cinetica de ley de potencia fraccionaria $n\in(0,1)$ y modulo de Thiele $\phi$. Para $\phi$ suficientemente grande (supercritico, $\phi>\phi^*$) aparece una **zona muerta** (*dead-core*) $[0,x_{dz}]$ donde $u\equiv0$, y la interfaz $x_{dz}$ es **desconocida a priori**.

La contribucion PINN del paper es un **ansatz duro (hard-constrained)** que incorpora el comportamiento asintotico exacto cerca de la interfaz directamente en la arquitectura, en vez de imponerlo como termino de perdida. Usando la coordenada transformada $\xi=(x-x_{dz})/(1-x_{dz})\in[0,1]$ (Eq. 19):

$$u(\xi)=\xi^p\big(1+(1-\xi)\mathcal{M}_\theta(\xi)\big)^2,\qquad p=\frac{2}{1-n}\quad\text{(Eq. 20, Listado 1.1)}$$

Esta construccion satisface **exactamente** $u(x_{dz})=0$, $u'(x_{dz})=0$ y $u(1)=1$ sin necesidad de terminos de perdida de contorno. La localizacion de la interfaz $x_{dz}$ se trata como un **parametro entrenable** mediante una parametrizacion sigmoide (Eq. 21, Listado 1.2): $x_{dz}=\sigma(\alpha)(1-\varepsilon)$, de forma que el gradiente fluye tanto hacia los pesos de la red como hacia $x_{dz}$. La perdida es unicamente el residuo fisico en el dominio transformado (Eq. 22-24, Listado 1.3):

$$\mathcal{R}(\xi)=\frac{1}{(1-x_{dz})^2}\frac{d^2u}{d\xi^2}-\phi^2u^n,\qquad \mathcal{L}_{PDE}=\mathbb{E}_{\xi\in[0,1]}[\mathcal{R}(\xi)^2]$$

con muestreo mixto: mitad uniforme, mitad sesgado hacia $\xi=0$ via $\xi_{biased}=U^\beta$, $\beta>1$ (Eq. 25), para resolver mejor la zona de transicion rapida cerca de la interfaz.

Este cuaderno reproduce **literalmente** el `forward` (Listado 1.1), la parametrizacion de $x_{dz}$ (Listado 1.2), el residuo (Listado 1.3) y la estrategia de muestreo (Eq. 25) tal como aparecen en el paper, y compara el resultado contra la solucion exacta con zona muerta (Eq. 13, 16 y la formula cerrada de la seccion 2.3) para el caso supercritico de la Fig. 1 del paper ($n=0.5$, $\phi=6.0$, $x_{dz}$ exacto $=0.42265$).

## Repositorio publico de referencia

El PDF no incluye un repositorio de codigo (aunque si incluye fragmentos de codigo directamente en el texto, Listados 1.1-1.3, que este cuaderno reproduce). Como referencia publica general de PINNs con restricciones duras (*hard-constrained ansatz*), se usa:

- **lululxvi/deepxde** &mdash; https://github.com/lululxvi/deepxde — biblioteca de referencia para PINNs que soporta restricciones duras de contorno mediante transformaciones de la salida de la red, el mismo principio usado en este paper.

In [ ]:
# Instalacion de dependencias (ejecutar si no estan ya instaladas en el entorno)
%pip install -q torch numpy matplotlib

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

torch.manual_seed(0)
np.random.seed(0)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## 1. Caso supercritico de la Fig. 1 del paper: $n=0.5$, $\phi=6.0$

In [ ]:
n_exp = 0.5
phi = 6.0
p_exp = 2.0 / (1.0 - n_exp)  # Eq. (18)

phi_star = np.sqrt(2 * (1 + n_exp)) / (1 - n_exp)  # Eq. (15), modulo de Thiele critico
x_dz_exact = 1 - phi_star / phi                     # Eq. (13)
print(f'p = {p_exp}, phi* = {phi_star:.4f}, x_dz exacto = {x_dz_exact:.5f} (paper reporta 0.42265)')

def exact_solution(x):
    """u(x) exacta con zona muerta (texto, Seccion 2.3, tras Eq. 16)."""
    u = np.zeros_like(x)
    mask = x > x_dz_exact
    u[mask] = ((x[mask] - x_dz_exact) / (1 - x_dz_exact)) ** (2 / (1 - n_exp))
    return u

## 2. Ansatz duro y parametrizacion de la interfaz $x_{dz}$ (Listados 1.1-1.2, literalmente del paper)

In [ ]:
class DeadCorePINN(nn.Module):
    def __init__(self, n_hidden=4, n_neurons=32, p=p_exp, eps=1e-4):
        super().__init__()
        layers = [nn.Linear(1, n_neurons), nn.Tanh()]
        for _ in range(n_hidden - 1):
            layers += [nn.Linear(n_neurons, n_neurons), nn.Tanh()]
        layers += [nn.Linear(n_neurons, 1)]
        self.net = nn.Sequential(*layers)
        self.p = p
        self.eps = eps
        # Listado 1.2: parametrizacion sigmoide entrenable de x_dz, Eq. (21)
        self.xdz_param = nn.Parameter(torch.tensor([0.0]))

    @property
    def xdz(self):
        return torch.sigmoid(self.xdz_param) * (1.0 - self.eps)

    def forward(self, xi):
        # Listado 1.1: ansatz duro u(xi) = xi^p * (1 + (1-xi)*M(xi))^2, Eq. (20)
        M = self.net(xi)
        B = 1.0 + (1.0 - xi) * M
        u = (xi ** self.p) * (B ** 2)
        return u


model = DeadCorePINN().to(device)

## 3. Residuo fisico y muestreo sesgado hacia la interfaz (Listado 1.3, Eq. 22-25)

In [ ]:
def sample_xi(n_points, beta=3.0):
    """Mitad uniforme, mitad sesgada hacia xi=0 via xi_biased = U^beta, Eq. (25)."""
    n_half = n_points // 2
    xi_uniform = torch.rand(n_half, 1)
    xi_biased = torch.rand(n_points - n_half, 1) ** beta
    xi = torch.cat([xi_uniform, xi_biased], dim=0).clamp(min=1e-4, max=1.0)
    return xi.to(device).requires_grad_(True)


def pde_loss(model, xi):
    u = model(xi)
    du_dxi = torch.autograd.grad(u, xi, grad_outputs=torch.ones_like(u),
                                  create_graph=True, retain_graph=True)[0]
    d2u_dxi2 = torch.autograd.grad(du_dxi, xi, grad_outputs=torch.ones_like(du_dxi),
                                    create_graph=True, retain_graph=True)[0]
    L = 1.0 - model.xdz
    u_xx = d2u_dxi2 / (L ** 2)  # Listado 1.3, Eq. (22)
    residual = u_xx - phi**2 * torch.clamp(u, min=0.0) ** n_exp  # Eq. (23)
    return torch.mean(residual**2)

## 4. Entrenamiento: la red y $x_{dz}$ se optimizan simultaneamente

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=2e-3)
history, xdz_history = [], []

for epoch in range(4000):
    optimizer.zero_grad()
    xi = sample_xi(400)
    loss = pde_loss(model, xi)
    loss.backward()
    optimizer.step()
    history.append(loss.item())
    xdz_history.append(model.xdz.item())
    if epoch % 500 == 0:
        print(f'epoch {epoch:5d} | loss={loss.item():.4e} | x_dz={model.xdz.item():.5f} '
              f'(exacto={x_dz_exact:.5f})')

## 5. Resultados: perfil de concentracion y localizacion de la zona muerta (cf. Fig. 1, panel derecho)

In [ ]:
xdz_final = model.xdz.item()
xi_plot = torch.linspace(1e-4, 1.0, 300, device=device).view(-1, 1)
with torch.no_grad():
    u_pinn = model(xi_plot).cpu().numpy().flatten()
x_plot = xdz_final + xi_plot.cpu().numpy().flatten() * (1 - xdz_final)  # x = x_dz + xi*(1-x_dz), Eq. (19)

x_full = np.linspace(0, 1, 300)
u_exact_full = exact_solution(x_full)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
axes[0].plot(x_full, u_exact_full, label='Solucion exacta (con zona muerta)', linewidth=2)
axes[0].plot(x_plot, u_pinn, '--', label='PINN (ansatz duro)')
axes[0].axvline(x_dz_exact, color='gray', linestyle=':', label=f'$x_{{dz}}$ exacto = {x_dz_exact:.3f}')
axes[0].axvline(xdz_final, color='red', linestyle=':', label=f'$x_{{dz}}$ PINN = {xdz_final:.3f}')
axes[0].set_xlabel('x'); axes[0].set_ylabel('u(x)')
axes[0].set_title(f'Perfil de concentracion (n={n_exp}, phi={phi}, supercritico)')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].semilogy(history)
axes[1].set_xlabel('Epoca'); axes[1].set_ylabel('Loss PDE (escala log)')
axes[1].set_title('Convergencia')
axes[1].grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f'x_dz identificado por la PINN: {xdz_final:.5f}')
print(f'x_dz exacto (Eq. 13):          {x_dz_exact:.5f}')
print(f'Error relativo: {abs(xdz_final - x_dz_exact) / x_dz_exact * 100:.2f}%')